In [18]:
import pandas as pd
import numpy as np

In [3]:
# Load the datasets
sales = pd.read_csv("../data/processed/online_retail_cleaned.csv", parse_dates=["InvoiceDate"])
returns = pd.read_csv("../data/processed/online_retail_returns.csv", parse_dates=["InvoiceDate"])

In [4]:
# 1) Number of rows, invoices, and identified customers
identified = sales[sales["CustomerID"] != -1]
print("Rows:", f"{len(sales):,}")
print("Invoices:", f"{sales['InvoiceNo'].nunique():,}")
print("Identified customers:", f"{identified['CustomerID'].nunique():,}")

# 2) Proportion of rows and revenue from unidentified customers (CustomerID == -1)
no_id = sales["CustomerID"] == -1
print(f"CustomerID == -1: {no_id.mean():.1%} of rows, "
      f"{sales.loc[no_id, 'TotalPrice'].sum() / sales['TotalPrice'].sum():.1%} of revenue")

# 3) date range of the dataset
print("Date range:", sales["InvoiceDate"].min(), "->", sales["InvoiceDate"].max())

# 4) largest sales by TotalPrice
cols = ["InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice", "CustomerID", "InvoiceDate", "TotalPrice"]
print(sales.nlargest(5, "TotalPrice")[cols])

# 5) Combine returns: C-invoices vs. non-C invoices
returns["IsCInvoice"] = returns["InvoiceNo"].astype(str).str.startswith("C")
print(returns["IsCInvoice"].value_counts())

Rows: 522,537
Invoices: 19,773
Identified customers: 4,334
CustomerID == -1: 25.1% of rows, 14.7% of revenue
Date range: 2010-12-01 08:26:00 -> 2011-12-09 12:50:00
        InvoiceNo StockCode                          Description  Quantity  \
521070     581483     23843          PAPER CRAFT , LITTLE BIRDIE     80995   
59407      541431     23166       MEDIUM CERAMIC TOP STORAGE JAR     74215   
214408     556444     22502       PICNIC BASKET WICKER 60 PIECES        60   
335866     567423     23243  SET OF TEA COFFEE SUGAR TINS PANTRY      1412   
50697      540815     21108   FAIRY CAKE FLANNEL ASSORTED COLOUR      3114   

        UnitPrice  CustomerID         InvoiceDate  TotalPrice  
521070       2.08       16446 2011-12-09 09:15:00   168469.60  
59407        1.04       12346 2011-01-18 10:01:00    77183.60  
214408     649.50       15098 2011-06-10 15:28:00    38970.00  
335866       5.06       17450 2011-09-20 11:05:00     7144.72  
50697        2.10       15749 2011-01-11 12:55:

In [5]:
cols = ["InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice", "CustomerID", "InvoiceDate", "TotalPrice"]

# 1) Largest rows of returns: Is there a pair of suspicious three rows there?
print(returns.nsmallest(5, "TotalPrice")[cols])

# 2) what are 474 rows without C?
non_c = returns[~returns["IsCInvoice"]]
print("Share with UnitPrice == 0:", (non_c["UnitPrice"] == 0).mean().round(3))
print("Share with CustomerID == -1:", (non_c["CustomerID"] == -1).mean().round(3))
print(non_c["StockCode"].value_counts().head(5))

# 3) Product vs. non-product returns (following the same Step 5 pattern)
is_product = returns["StockCode"].astype(str).str.match(r"^(?:\d{5}|DCGS)")
print(returns.groupby(is_product)["TotalPrice"].agg(["count", "sum"]))

     InvoiceNo  StockCode                     Description  Quantity  \
9718   C581484      23843     PAPER CRAFT , LITTLE BIRDIE    -80995   
1045   C541433      23166  MEDIUM CERAMIC TOP STORAGE JAR    -74215   
4143   C556445          M                          Manual        -1   
9484   C580605  AMAZONFEE                      AMAZON FEE        -1   
744    C540117  AMAZONFEE                      AMAZON FEE        -1   

      UnitPrice  CustomerID         InvoiceDate  TotalPrice  
9718       2.08       16446 2011-12-09 09:27:00  -168469.60  
1045       1.04       12346 2011-01-18 10:17:00   -77183.60  
4143   38970.00       15098 2011-06-10 15:31:00   -38970.00  
9484   17836.46          -1 2011-12-05 11:36:00   -17836.46  
744    16888.02          -1 2011-01-05 09:55:00   -16888.02  
Share with UnitPrice == 0: 1.0
Share with CustomerID == -1: 1.0
StockCode
85175     5
85172     4
21830     4
72802C    4
22423     3
Name: count, dtype: int64
           count        sum
StockCode    

## Analysis-ready data

In [9]:
# Convert CustomerID to nullable integer type
sales['CustomerID'] = sales['CustomerID'].astype('Int64').mask(sales['CustomerID'] == -1, pd.NA)

In [ ]:
# Invoices that were fully reversed by a cancellation invoice within minutes.
# Evidence: each has a matching "C" invoice with the same CustomerID and the same
# total amount (168,469.60 / 77,183.60 / 38,970.00), issued 3-16 minutes later.
# Note: for 556444 the cancellation is a manual line (StockCode "M", Quantity -1),
# so it matches by amount, not by product or quantity.
# Assumption (not proven): these are data-entry corrections, not real customer returns.
# Found by inspecting the largest sales lines; no general matching was attempted.
REVERSED_INVOICES = [581483, 541431, 556444]

sales["IsReversedOrder"] = sales["InvoiceNo"].isin(REVERSED_INVOICES)

# value counts and total revenue of reversed orders
sales.groupby("IsReversedOrder")["TotalPrice"].agg(["count", "sum"])

,count,sum
IsReversedOrder,,
False,522534,9962596.12
True,3,284623.20


In [ ]:
# Replace 'EIRE' with 'Ireland' in the Country column
returns['Country'] = returns['Country'].replace('EIRE', 'Ireland')

In [ ]:
# Categorize return types based on StockCode and IsCInvoice
cond_product_cancel = returns["StockCode"].astype(str).str.match(r"^(?:\d{5}|DCGS)") & returns["IsCInvoice"]
cond_non_product_cancel = ~returns["StockCode"].astype(str).str.match(r"^(?:\d{5}|DCGS)") & returns["IsCInvoice"]

conditions = [cond_product_cancel,
               cond_non_product_cancel]

choices = ["product_cancellation",
           "non_product_cancellation"]

returns["ReturnType"] = np.select(conditions, choices, default="stock_adjustment")

In [26]:
returns.groupby("ReturnType")["TotalPrice"].agg(["count", "sum"])

,count,sum
ReturnType,,
non_product_cancellation,583,-418078.57
product_cancellation,8668,-475901.16
stock_adjustment,474,0.00
